# Reddit Sentiment Scraper - Enhanced Version

This notebook uses the **enhanced** functions from `utile_reddit.py` to collect and analyze Reddit sentiment data with improved coverage.

**Key Enhancements:**
- **31 subreddits** (vs 11 original) for broader coverage
- **Enhanced ticker-specific subreddits** for low-coverage tickers (MRVL, AEM, APP, SMR, MU, VERU)
- **Improved regex patterns** for better ticker matching
- **Higher post limits** for low-coverage tickers (200 vs 120)
- **Fixed aggregation functions** (no more errors)
- **Enhanced reporting** with detailed coverage analysis

**Expected Results:**
- 2-3x more data for previously low-coverage tickers
- Better temporal coverage across all months
- Stronger sentiment signals with more data points


In [ ]:
# Setup: Import enhanced functions and set Reddit API credentials
import os
from utile_reddit import run_pipeline_classic, EXPANDED_SUBS, TICKER_SUB_MAP_IMPROVED, TICKER_SPECIFIC_LIMITS

# Reddit API credentials
os.environ['REDDIT_CLIENT_ID'] = "pLqfk1M1ymfj3ih1NrVFlA"
os.environ['REDDIT_CLIENT_SECRET'] = "_hl1434FeTi9kgv_GXAi5tBLoCaLIQ"
os.environ['REDDIT_USER_AGENT'] = "SentimentAnalysisBot"

print("✅ Enhanced Reddit sentiment pipeline loaded")
print("✅ Reddit API credentials configured")
print(f"✅ {len(EXPANDED_SUBS)} subreddits available for enhanced coverage")


In [ ]:
# Configuration: Enhanced setup for better coverage
TICKERS = [
    "NVDA", "AMD", "MSFT", "ASML", "GOOG", "GOOGL", "PLTR", "MRVL", "APP",
    "MU", "IONQ", "RGTI", "QBTS", "SMR", "AI", "RDDT", "ARBE", "AEM", "VERU", "QQQ",
]

print(f"🚀 ENHANCED CONFIGURATION:")
print(f"   Tickers: {len(TICKERS)}")
print(f"   General subreddits: {len(EXPANDED_SUBS)} (auto-loaded)")
print(f"   Date range: 2015 to present")
print()
print(f"💡 IMPROVEMENTS FOR LOW-COVERAGE TICKERS:")
low_coverage = ['MRVL', 'AEM', 'APP', 'SMR', 'MU', 'VERU']
for ticker in low_coverage:
    subs = TICKER_SUB_MAP_IMPROVED.get(ticker, [])
    limit = TICKER_SPECIFIC_LIMITS.get(ticker, 120)
    print(f"   {ticker}: {len(subs)} subreddits, {limit} posts/month")
print()
print(f"📈 Expected: 2-3x more data for low-coverage tickers!")


In [ ]:
# Run the Enhanced Reddit Sentiment Pipeline
# This automatically uses:
# - 31 subreddits (vs 11 original)
# - Enhanced ticker-specific subreddits
# - Improved regex patterns
# - Higher limits for low-coverage tickers
# - Fixed aggregation (no more errors!)

monthly = run_pipeline_classic(
    tickers=TICKERS,
    subs=None,  # Uses EXPANDED_SUBS automatically
    start_year=2015,
    end_year=None,  # Collect to present
    per_query_limit=120,  # Will use higher limits for specific tickers
    include_top_comments=True,
    top_comments_k=5,
    model="reddit_plus",  # Best performing model
    out_dir="out/reddit",
    use_expanded_subs=True  # Enable all enhancements!
)

print("\n🎯 PIPELINE COMPLETED WITH ENHANCED CONFIGURATION!")
print("📁 Check out/reddit/ folder for enhanced results")


In [ ]:
# Quick validation: Compare with previous results
import pandas as pd

# Load the enhanced results
monthly = pd.read_csv('out/reddit/reddit_monthly_sentiment.csv')

print(f"📊 ENHANCED RESULTS SUMMARY:")
print(f"   Monthly observations: {len(monthly):,}")
print(f"   Tickers covered: {monthly['ticker'].nunique()}")
print(f"   Date range: {monthly['month'].min()} to {monthly['month'].max()}")
print(f"   Sentiment coverage: {monthly['sent_mean'].notna().mean():.1%}")

# Focus on previously low-coverage tickers
low_coverage = ['MRVL', 'AEM', 'APP', 'SMR', 'MU', 'VERU']
improved_data = monthly[monthly['ticker'].isin(low_coverage)]

print(f"\n🚀 IMPROVEMENT ANALYSIS (Low-Coverage Tickers):")
for ticker in low_coverage:
    ticker_data = improved_data[improved_data['ticker'] == ticker]
    if not ticker_data.empty:
        total_posts = ticker_data['n_posts'].sum()
        months_covered = len(ticker_data)
        avg_sentiment = ticker_data['sent_mean'].mean()
        print(f"   {ticker}: {total_posts:,} posts, {months_covered} months, {avg_sentiment:.3f} avg sentiment")
    else:
        print(f"   {ticker}: No data found")

# Top performers
print(f"\n🏆 TOP PERFORMERS BY POST COUNT:")
top_tickers = monthly.groupby('ticker')['n_posts'].sum().sort_values(ascending=False).head(10)
for ticker, posts in top_tickers.items():
    print(f"   {ticker}: {posts:,} total posts")

print(f"\n✅ Enhanced configuration delivered improved data collection!")


In [2]:
# Define your ticker universe here. These tickers will be used for
# matching posts and generating sentiment metrics.
TICKERS = [
    "NVDA", "AMD", "MSFT", "ASML", "GOOG", "GOOGL", "PLTR", "MRVL", "APP",
    "MU", "IONQ", "RGTI", "QBTS", "SMR", "AI", "RDDT", "ARBE", "AEM", "VERU", "QQQ",
]

# Core finance subreddits plus a few niche ones. You can add or
# remove subreddits here to broaden or narrow the scope of the
# collection.
SUBS = [
    "stocks", "investing", "stockmarket", "financialnews", "quant",
    "pennystocks", "biotechstocks", "biotech", "mining", "gold", "canadianinvestor",
]

# Start and end years define the date range for collection. The end
# year can be None to collect up to the present.
START_YEAR = 2015
END_YEAR = None

# Collection mode settings. Only `scroll` is used in this script,
# but search mode is included for completeness.
COLLECTION_MODE = "scroll"
USE_CLOUDSEARCH = True
USE_MULTI_SUB_SEARCH = True
SEARCH_PAGE_LIMIT = 1000

# Per (ticker, month) cap to avoid a single viral thread
# dominating the sentiment. Adjust this if you find certain
# tickers overwhelm the dataset with too many posts.
PER_QUERY_LIMIT = 120

# Matching / false‑positive control.
STRICT_MATCH = True  # use brand/ticker regex
INCLUDE_TICKER_SUBS = True  # add r/amd, r/nvidia, etc.
DYNAMIC_DISCOVERY = False  # auto‑discover new subs (not implemented here)

# Company context filter (e.g. remove Google product chatter). Put
# tickers here to enable the filter; leave empty to disable.
APPLY_COMPANY_CONTEXT_FOR = {"GOOG", "GOOGL"}

# Text eligibility for sentiment.
SCORE_ONLY_REDDIT_TEXT = True
MIN_HEADLINE_CHARS = 10  # shorten from 15 to allow more titles
MIN_TOTAL_CHARS = 30  # reduce total chars to allow comments to carry sentiment
INCLUDE_TOP_COMMENTS = True
TOP_COMMENTS_K = 5
MAX_ST_CHARS = 1500
MAX_COMMENTS_CHARS = 1200

# Sentiment model settings.
MODEL = "reddit_plus"  # options: reddit, finance, bertweet, ensemble, reddit_plus
ENSEMBLE_WEIGHTS = (0.4, 0.6)
USE_VADER = True
VADER_WEIGHT = 0.12
ALPHA_COMMENTS = 0.5

# Performance knobs.
REQUEST_TIMEOUT = 20
BATCH_SIZE = 32
MAX_SEQ_LEN = 256
USE_FP16 = True
SLEEP_BETWEEN_MONTHS = 0.15

# Output directory for CSV files.
OUT_DIR = "out/reddit"

# ============================
# Regex builder
# ============================

def build_ticker_regex():
    """
    Build a dictionary mapping tickers to compiled regular expressions
    that match various forms of the ticker in text. Patterns include
    brand/alias terms as well as stock‑notation expansions:

      * Cashtags like `$NVDA`
      * Exchange prefixes such as `NASDAQ: NVDA`
      * Dot suffixes for selected tickers (e.g. `AEM.TO`)

    These patterns are designed to catch mentions in general finance
    subreddits where users often use shorthand or stock notation.

    Returns
    -------
    dict
        A mapping from ticker to compiled regex pattern.
    """
    base_terms = {
        'AEM':   r"(agnico(?:\s+eagle(?:\s+mines)?)?|agnico)",
        'AI':    r"(c3\.ai|c3ai|c3\s*ai)",
        'AMD':   r"(amd|advanced\s+micro\s+devices|ryzen|epyc|radeon)",
        'APP':   r"(app[- ]?lovin|applovin|app\s*lovin\s*corp)",
        'ARBE':  r"(arbe\s+robotics|arbe)",
        'ASML':  r"(asml|asml\s+holding|extreme\s+ultraviolet|euv)",
        'GOOG':  r"(google|alphabet|goog|googl)",
        'GOOGL': r"(google|alphabet|googl|goog)",
        'IONQ':  r"(ionq|ionq\s+quantum)",
        'MRVL':  r"(marvell|mrvl)",
        'MSFT':  r"(msft|microsoft|azure|xbox)",
        'MU':    r"(micron|micron\s+technology)",
        'NVDA':  r"(nvidia|nvda|cuda|geforce|tensor\s*core)",
        'PLTR':  r"(palantir|pltr)",
        'QBTS':  r"(d[- ]?wave|qbts)",
        'QQQ':   r"(qqq|invesco\s+nasdaq\s*100)",
        'RDDT':  r"(reddit|rddt)",
        'RGTI':  r"(rigetti|rgti)",
        'SMR':   r"(nu\s*scale|nuscale|smr|small\s+modular\s+reactor)",
        'VERU':  r"(veru(?:\s+pharma)?|veru\s+inc)",
    }

    exch_prefixes = (
        "nasdaq", "nyse", "nysearca", "amex", "tsx", "tsxv", "cboe", "otc"
    )
    suffixes_by_ticker: Dict[str, Tuple[str, ...]] = {
        'AEM': ("to", "tsx"),  # AEM.TO / AEM.TSX
        # Add more suffixes here if other tickers trade with dots
    }

    def _stock_notations(tkr: str) -> str:
        t = re.escape(tkr)
        cashtag = rf"\${t}\b"
        exch = rf"(?:{'|'.join(exch_prefixes)})\s*:\s*{t}\b"
        sfx = suffixes_by_ticker.get(tkr, ())
        dot = rf"{t}\.(?:{'|'.join(map(re.escape, sfx))})\b" if sfx else None
        parts = [cashtag, exch] + ([dot] if dot else [])
        return "(?:" + "|".join(parts) + ")"

    compiled: Dict[str, Pattern[str]] = {}
    for tkr, brand_rx in base_terms.items():
        brand = rf"(?:{brand_rx})"
        notations = _stock_notations(tkr)
        pat = rf"(?:{brand}|{notations})"
        compiled[tkr] = re.compile(pat, flags=re.I)
    return compiled

# ============================
# Ticker sub map
# ============================

# Known ticker‑specific subreddits. These are used to match posts
# where the ticker might not appear explicitly in the post text.
# Remove dead subs or add new ones as needed.
TICKER_SUB_MAP_DEFAULT: Dict[str, List[str]] = {
    'AMD': ['amd', 'amd_stock'],
    'NVDA': ['nvidia', 'nvda_stock'],
    'ASML': ['asml'],
    'MRVL': ['marvell'],
    'MU': ['micron'],
    'MSFT': ['microsoft'],
    'GOOG': ['alphabetinc'],  # avoid r/google (product chatter)
    'GOOGL': ['alphabetinc'],
    'PLTR': ['palantir'],
    'APP': ['applovin'],
    'AI': ['c3ai'],
    'IONQ': ['ionq'],
    'RGTI': ['rigetti'],
    'QBTS': ['dwave', 'qbtsstock'],
    'SMR': ['nuscale'],
    'RDDT': ['rddt'],
    'QQQ': ['qqq'],
    'ARBE': ['arbe_robotics', 'arbe_investors'],
    'VERU': ['verupharma'],
    'AEM': ['agnicoeagle'],
}

# ============================
# PRAW client & helpers
# ============================

MEDIA_BLOCKLIST = {
    'i.redd.it', 'v.redd.it', 'i.imgur.com', 'imgur.com',
    'gfycat.com', 'youtube.com', 'youtu.be'
}

def _praw_client(request_timeout: int = REQUEST_TIMEOUT) -> praw.Reddit:
    """Initialise a PRAW Reddit client with environment credentials."""
    cid = os.getenv("REDDIT_CLIENT_ID")
    csec = os.getenv("REDDIT_CLIENT_SECRET")
    ua = os.getenv("REDDIT_USER_AGENT")
    if not cid or not csec:
        raise RuntimeError(
            "Missing Reddit API credentials. Set REDDIT_CLIENT_ID and REDDIT_CLIENT_SECRET."
        )
    return praw.Reddit(
        client_id=cid,
        client_secret=csec,
        user_agent=ua,
        requestor_kwargs={'timeout': request_timeout},
        check_for_async=False,
    )

def _domain(url: str) -> str:
    """Extract the domain from a URL, returning lowercase."""
    try:
        return urlparse(url).netloc.lower()
    except Exception:
        return ''

def _month_bounds(year: int, month: int) -> Tuple[pd.Timestamp, pd.Timestamp]:
    """Return the start and end timestamps (UTC) of a month."""
    start = pd.Timestamp(year=year, month=month, day=1, tz='UTC')
    end = (start + pd.offsets.MonthEnd(1)).normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
    return start, end

# ============================
# Collector (scroll only)
# ============================

def collect_reddit_monthly_scroll(
    tickers: Sequence[str],
    start_year: int,
    end_year: Optional[int],
    subreddits: Sequence[str],
    include_ticker_subs: bool = True,
    ticker_sub_map: Optional[Dict[str, List[str]]] = None,
    per_query_limit: int = 120,
    strict_match: bool = True,
    ticker_regex: Optional[Dict[str, Pattern[str]]] = None,
    include_top_comments: bool = False,
    top_comments_k: int = 3,
    request_timeout: int = REQUEST_TIMEOUT,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Scroll through subreddits and collect posts mentioning given tickers.

    This collector iterates through the `new` listing of each subreddit,
    matches posts either by regex (brand terms and stock notations) or by
    the subreddit name if it belongs to a ticker, and fetches top
    comments for posts with at least one comment. It stores per‑post
    metadata and returns a DataFrame.

    Parameters
    ----------
    tickers : Sequence[str]
        List of tickers to search for.
    start_year : int
        Starting year for collection (inclusive).
    end_year : int or None
        Ending year for collection. If None, collects up to the
        current date.
    subreddits : Sequence[str]
        List of subreddits to scroll through.
    include_ticker_subs : bool, optional
        Whether to include known ticker‑specific subreddits.
    ticker_sub_map : dict, optional
        Mapping of tickers to their associated subreddits. If not
        provided, uses `TICKER_SUB_MAP_DEFAULT`.
    per_query_limit : int, optional
        Maximum number of posts to keep per (ticker, month).
    strict_match : bool, optional
        If True, require regex match for text; if False, any post in
        the ticker subreddit counts.
    ticker_regex : dict, optional
        Precompiled regex patterns for each ticker.
    include_top_comments : bool, optional
        Whether to fetch top comments.
    top_comments_k : int, optional
        Number of top comments to keep per post.
    request_timeout : int, optional
        HTTP timeout for Reddit requests.
    verbose : bool, optional
        If True, prints progress messages.

    Returns
    -------
    pandas.DataFrame
        A DataFrame with per‑post metadata.
    """
    # Normalise tickers and subs
    if isinstance(tickers, str):
        tickers = [t.strip().upper() for t in tickers.split(',') if t.strip()]
    else:
        tickers = [str(t).strip().upper() for t in tickers if str(t).strip()]
    if isinstance(subreddits, str):
        subreddits = [s.strip().lower() for s in subreddits.split(',') if s.strip()]
    else:
        subreddits = [str(s).strip().lower() for s in subreddits if str(s).strip()]

    reddit = _praw_client(request_timeout=request_timeout)
    tregex = ticker_regex or build_ticker_regex()
    tsubs = ticker_sub_map or TICKER_SUB_MAP_DEFAULT

    # Build the set of subreddits to scroll
    base_subs: Set[str] = set(subreddits)
    if include_ticker_subs:
        for t in tickers:
            base_subs |= set(tsubs.get(t, []))
    subs = sorted(base_subs)

    # Reverse map: subreddit -> tickers. Posts in ticker subs count even
    # if the text does not match the regex.
    sub_to_tickers: Dict[str, Set[str]] = defaultdict(set)
    if include_ticker_subs and tsubs:
        for tkr, sublist in tsubs.items():
            for s in sublist:
                sub_to_tickers[s.lower()].add(tkr.upper())

    earliest = pd.Timestamp(year=start_year, month=1, day=1, tz='UTC')
    latest = pd.Timestamp.utcnow() if end_year is None else pd.Timestamp(year=end_year, month=12, day=31, tz='UTC')

    rows: List[Dict] = []
    kept: Dict[Tuple[str, str], int] = {}
    if verbose:
        print(f"[Scroll] {len(subs)} subs | window {earliest.date()}..{latest.date()}")
        print(f"[Scroll] Tickers: {', '.join(tickers)}")

    for sub in subs:
        try:
            sr = reddit.subreddit(sub)
            n_before = len(rows)
            for post in sr.new(limit=None):
                ts = getattr(post, 'created_utc', None)
                if ts is None:
                    continue
                created = pd.to_datetime(ts, unit='s', utc=True)
                if created > latest:
                    continue
                if created < earliest:
                    break  # We scrolled past our lower bound

                title = (getattr(post, 'title', '') or '').strip()
                selftext = (getattr(post, 'selftext', '') or '').strip()
                headline = (title + ' ' + selftext).strip() if selftext else title

                # Match tickers via regex and subreddit mapping
                matched = set()
                for tkr in tickers:
                    if strict_match and tkr in tregex and not tregex[tkr].search(headline):
                        continue
                    matched.add(tkr)
                sname = str(getattr(post, 'subreddit', '')).lower()
                if sname in sub_to_tickers:
                    for tkr in sub_to_tickers[sname]:
                        if tkr in tickers:
                            matched.add(tkr)
                if not matched:
                    continue

                permalink = f"https://www.reddit.com{getattr(post, 'permalink', '')}"
                ext_url = getattr(post, 'url', '') or permalink
                dom = _domain(ext_url) or 'reddit.com'
                is_media = dom in MEDIA_BLOCKLIST
                is_reddit = dom.endswith('reddit.com') or dom.endswith('redd.it')

                score = int(getattr(post, 'score', 0) or 0)
                num_comments = int(getattr(post, 'num_comments', 0) or 0)

                comments_joined = ""
                # Fetch top comments if available
                if include_top_comments and num_comments >= 1:
                    try:
                        submission = reddit.submission(id=getattr(post, 'id'))
                        # Try 'top' first, then 'best'
                        for sort in ('top', 'best'):
                            try:
                                submission.comment_sort = sort
                                submission.comment_limit = 80
                                submission.comments.replace_more(limit=16)
                                comments = [c for c in submission.comments if isinstance(getattr(c, 'body', None), str)]
                                comments.sort(key=lambda c: (c.score or 0), reverse=True)
                                topbodies = [
                                    c.body.strip() for c in comments[:top_comments_k]
                                    if c.body and len(c.body.strip()) > 8
                                ]
                                if topbodies:
                                    comments_joined = "\n\n".join(topbodies)
                                    break
                            except Exception:
                                continue
                    except Exception:
                        comments_joined = ""

                # Compute month key (period) for the post
                mkey = created.tz_convert('UTC').tz_localize(None).to_period('M').strftime('%Y-%m')
                for tkr in matched:
                    cap_key = (tkr, mkey)
                    if kept.get(cap_key, 0) >= per_query_limit:
                        continue
                    rows.append({
                        'ticker': tkr,
                        'created_utc': float(ts),
                        'date': created,
                        'subreddit': sname,
                        'score': score,
                        'num_comments': num_comments,
                        'headline': title,
                        'selftext': selftext,
                        'top_comments': comments_joined,
                        'link': permalink,
                        'domain': dom,
                        'is_media': is_media,
                        'is_reddit': is_reddit,
                        'source': 'reddit_scroll',
                    })
                    kept[cap_key] = kept.get(cap_key, 0) + 1
        except Exception as e:
            if verbose:
                print(f"[Scroll] r/{sub} error: {e}")
        if verbose and len(rows) == n_before:
            print(f"[Scroll] r/{sub}: no matches in window (or all filtered)")

    df = pd.DataFrame(rows)
    if not df.empty:
        df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')
        # Canonicalise tickers: GOOG/GOOGL -> GOOG
        df['ticker'] = df['ticker'].replace({'GOOGL': 'GOOG'})
        df = (df.dropna(subset=['ticker', 'date'])
                .sort_values(['date', 'score'], ascending=[True, False])
                .drop_duplicates(subset=['link', 'ticker'])
                .reset_index(drop=True))
    return df

# ============================
# Sentiment text builder
# ============================

def make_sentiment_set(
    df: pd.DataFrame,
    min_headline_chars: int = MIN_HEADLINE_CHARS,
    min_total_chars: int = MIN_TOTAL_CHARS,
    include_top_comments: bool = True,
    max_selftext_chars: int = MAX_ST_CHARS,
    max_comments_chars: int = MAX_COMMENTS_CHARS,
) -> pd.DataFrame:
    """
    Construct the text used for sentiment scoring and determine
    eligibility. Posts become eligible if either the title or comments
    meet length requirements and the combined text is long enough.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing collected posts.
    min_headline_chars : int, optional
        Minimum length of the title for it to be considered.
    min_total_chars : int, optional
        Minimum total length of the combined text for eligibility.
    include_top_comments : bool, optional
        Whether to append top comments to the text.
    max_selftext_chars : int, optional
        Maximum length of selftext to include.
    max_comments_chars : int, optional
        Maximum length of comments to include.

    Returns
    -------
    pd.DataFrame
        DataFrame with new columns: `text_for_sentiment`,
        `sent_source`, and `is_sentiment_eligible`.
    """
    if df.empty:
        return df.assign(text_for_sentiment="", is_sentiment_eligible=False, sent_source="none")

    title = df['headline'].fillna('').astype(str)
    body = df['selftext'].fillna('').astype(str).str.slice(0, max_selftext_chars)
    if include_top_comments and 'top_comments' in df.columns:
        comm = df['top_comments'].fillna('').astype(str).str.slice(0, max_comments_chars)
    else:
        comm = pd.Series([''] * len(df), index=df.index)

    # Determine if title or comments are substantive
    title_ok = (title.str.len() >= min_headline_chars) | (title.str.split().str.len() >= 6)
    comments_ok = (comm.str.len() >= 60) | (comm.str.split().str.len() >= 12)

    text_list: List[str] = []
    source_list: List[str] = []
    for t, b, c in zip(title, body, comm):
        parts = []
        if t:
            parts.append(t)
        if b:
            parts.append(b)
        if include_top_comments and c:
            parts.append(c)
        text = "\n\n".join(parts).strip()
        text_list.append(text)
        src = []
        if t:
            src.append("title")
        if b:
            src.append("selftext")
        if include_top_comments and c:
            src.append("comments")
        source_list.append("+".join(src) if src else "none")

    out = df.copy()
    out['text_for_sentiment'] = text_list
    out['sent_source'] = source_list
    total_len = out['text_for_sentiment'].str.len()
    out['is_sentiment_eligible'] = (total_len >= min_total_chars) & (title_ok | comments_ok)
    return out

# ============================
# Company context filter
# ============================

FINANCE_WORDS: Set[str] = {
    'earnings', 'eps', 'revenue', 'guidance', 'profit', 'loss', 'margin', 'fcf',
    'valuation', 'multiple', 'pe', 'dividend', 'buyback', 'repurchase', 'split',
    'sec', '10-k', '10q', 'form 10-k', 'analyst', 'downgrade', 'upgrade',
    'target', 'price target', 'share', 'shares', 'stock', 'ticker', 'market cap',
}

PRODUCT_WORDS_MAP: Dict[str, Set[str]] = {
    'GOOG': {
        'maps','gmail','photos','chrome','pixel','android','nest','drive',
        'calendar','assistant','docs','sheets','slides','meet','chat','home','wifi','play','youtube music'
    },
    'GOOGL': {
        'maps','gmail','photos','chrome','pixel','android','nest','drive',
        'calendar','assistant','docs','sheets','slides','meet','chat','home','wifi','play','youtube music'
    },
}

def apply_company_context_filter(
    df,
    only_company_for = APPLY_COMPANY_CONTEXT_FOR,
    finance_words = FINANCE_WORDS,
    product_words_map = PRODUCT_WORDS_MAP,
    finance_subs: Tuple[str, ...] = tuple(SUBS),
) -> pd.DataFrame:
    """
    Filter out posts that are clearly about products rather than the
    company/stock for certain tickers. This is especially useful for
    GOOG/GOOGL, where many posts discuss Google products like Android
    or Maps without referencing the stock.

    A post is kept if it meets at least one of the following:
    * It does not belong to a ticker in `only_company_for`.
    * Its text contains finance keywords.
    * It appears in a finance subreddit and does not contain product
      keywords.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame of collected posts.
    only_company_for : set of str, optional
        Tickers for which to apply the filter.
    finance_words : set of str, optional
        Keywords indicating a finance context.
    product_words_map : dict, optional
        Mapping from tickers to sets of product keywords.
    finance_subs : tuple, optional
        Finance subreddits; posts in these subs are treated as company
        context unless they contain product keywords.

    Returns
    -------
    pd.DataFrame
        Filtered DataFrame.
    """
    if df.empty or not only_company_for:
        return df

    text = (df['headline'].fillna('') + ' ' + df['selftext'].fillna('')).str.lower()

    keep = pd.Series(True, index=df.index)
    if any(ticker in only_company_for for ticker in df['ticker'].unique()):
        fin_re = re.compile(r'(' + '|'.join(map(re.escape, finance_words)) + ')', flags=re.I)
        for tkr in only_company_for:
            mask = df['ticker'] == tkr
            if not mask.any():
                continue
            prod_words = product_words_map.get(tkr, set())
            prod_re = re.compile(r'(' + '|'.join(map(re.escape, prod_words)) + ')', flags=re.I) if prod_words else None
            text_mask = text[mask]
            has_fin = text_mask.str.contains(fin_re, regex=True)
            has_prod = text_mask.str.contains(prod_re, regex=True) if prod_re else False
            in_fin_sub = df.loc[mask, 'subreddit'].isin(finance_subs)
            keep_loc = has_fin | (in_fin_sub & ~has_prod)
            keep.loc[mask] = keep_loc
    return df.loc[keep].reset_index(drop=True)

# ============================
# Sentiment scoring
# ============================

URL_RE = re.compile(r"https?://\S+|www\.\S+", re.I)
USER_RE = re.compile(r"(?<!\w)u/[A-Za-z0-9_-]+|@[A-Za-z0-9_]+", re.I)
HASHTAG_RE = re.compile(r"#(\w+)")
WS_RE = re.compile(r"\s+")

def _clean_social_text(texts: Iterable[str]) -> List[str]:
    out: List[str] = []
    for t in texts:
        t = t or ''
        t = URL_RE.sub(' http ', t)
        t = USER_RE.sub(' @user ', t)
        t = HASHTAG_RE.sub(lambda m: m.group(1), t)
        t = t.replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>')
        t = WS_RE.sub(' ', t).strip()
        out.append(t)
    return out

def _intensity_heuristic(texts: Iterable[str]) -> np.ndarray:
    """Compute a bounded intensity multiplier based on exclamation marks, all caps, and elongated words."""
    factors: List[float] = []
    caps_re = re.compile(r"\b[A-Z]{3,}\b")
    elong_re = re.compile(r"(.)\1{2,}")
    for t in texts:
        t = t or ""
        n_caps = len(caps_re.findall(t))
        n_bang = t.count('!')
        n_elong = len(elong_re.findall(t.lower()))
        raw = 0.04*n_caps + 0.03*n_bang + 0.05*n_elong
        factors.append(float(np.clip(1.0 + raw, 0.9, 1.25)))
    return np.asarray(factors, dtype=np.float32)

def _engagement_boost(scores: np.ndarray, ups: np.ndarray, ncom: np.ndarray, alpha_comments: float = ALPHA_COMMENTS) -> np.ndarray:
    w = np.log1p(np.clip(ups, 0, None)) + alpha_comments * np.log1p(np.clip(ncom, 0, None))
    w = 1.0 + 0.10 * (1.0 - np.exp(-w / 5.0))
    return (scores * w).astype(np.float32)

@dataclass
class ModelChoice:
    name: str
    neg_label: Optional[str] = None
    pos_label: Optional[str] = None

MODEL_REGISTRY: Dict[str, ModelChoice] = {
    'finance': ModelChoice(name='ProsusAI/finbert', neg_label='negative', pos_label='positive'),
    'reddit': ModelChoice(name='cardiffnlp/twitter-roberta-base-sentiment-latest', neg_label='negative', pos_label='positive'),
    'bertweet': ModelChoice(name='finiteautomata/bertweet-base-sentiment-analysis', neg_label='NEG', pos_label='POS'),
}

@functools.lru_cache(maxsize=4)
def _load_model(name: str):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSequenceClassification.from_pretrained(name)
    return tok, mdl

def _predict_scores(
    texts: Iterable[str],
    name: str,
    neg_label: Optional[str],
    pos_label: Optional[str],
    batch_size: int = BATCH_SIZE,
    max_length: int = MAX_SEQ_LEN,
    use_fp16: bool = USE_FP16,
) -> np.ndarray:
    import torch
    tok, mdl = _load_model(name)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    mdl.to(device)
    mdl.eval()
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
    except Exception:
        pass
    scores: List[float] = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            batch = list(texts)[i:i+batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
            enc = {k: v.to(device) for k, v in enc.items()}
            if device.type == 'cuda' and use_fp16:
                with torch.autocast('cuda', dtype=torch.float16):
                    logits = mdl(**enc).logits
            else:
                logits = mdl(**enc).logits
            probs = logits.softmax(dim=-1).detach().cpu().numpy()
            if hasattr(mdl.config, 'id2label') and neg_label and pos_label:
                id2label = {int(k): v for k, v in mdl.config.id2label.items()}
                label2id = {v.lower(): k for k, v in id2label.items()}
                if neg_label.lower() in label2id and pos_label.lower() in label2id:
                    ni, pi = label2id[neg_label.lower()], label2id[pos_label.lower()]
                    s = probs[:, pi] - probs[:, ni]
                else:
                    s = probs[:, -1] - probs[:, 0]
            else:
                s = probs[:, -1] - probs[:, 0]
            scores.extend(s.tolist())
    return np.asarray(scores, dtype=np.float32)

def _vader_scores(texts: Iterable[str]) -> np.ndarray:
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    except Exception:
        return np.zeros(len(list(texts)), dtype=np.float32)
    an = SentimentIntensityAnalyzer()
    return np.asarray([an.polarity_scores(t or "")['compound'] for t in texts], dtype=np.float32)

def score_sentiment(
    df: pd.DataFrame,
    text_col: str = 'text_for_sentiment',
    mode: str = MODEL,
    ensemble_weights: Tuple[float, float] = ENSEMBLE_WEIGHTS,
    batch_size: int = BATCH_SIZE,
    max_seq_len: int = MAX_SEQ_LEN,
    use_fp16: bool = USE_FP16,
    use_vader: bool = USE_VADER,
    vader_weight: float = VADER_WEIGHT,
    alpha_comments: float = ALPHA_COMMENTS,
) -> pd.DataFrame:
    """
    Score sentiment for eligible rows in the DataFrame. Only rows
    previously marked as sentiment‑eligible are scored. Modes include
    `reddit`, `finance`, `bertweet`, `ensemble`, and `reddit_plus`.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing posts and a `is_sentiment_eligible` column.
    text_col : str, optional
        Name of the column containing text for sentiment.
    mode : str, optional
        Which sentiment model to use.
    ensemble_weights : tuple, optional
        Weights for the `finance` and `reddit` models if `mode` is
        `ensemble`.
    batch_size : int, optional
        Batch size for model inference.
    max_seq_len : int, optional
        Maximum sequence length for tokenisation.
    use_fp16 : bool, optional
        Whether to use mixed precision on CUDA.
    use_vader : bool, optional
        Whether to blend VADER scores for extra spread.
    vader_weight : float, optional
        Weight given to VADER when blending.
    alpha_comments : float, optional
        Engagement weighting parameter for the `reddit_plus` mode.

    Returns
    -------
    pd.DataFrame
        DataFrame with a new `sent_score` column.
    """
    if df.empty or not df.get('is_sentiment_eligible', pd.Series(dtype=bool)).any():
        df = df.copy()
        df['sent_score'] = []
        return df

    texts_raw = df.loc[df['is_sentiment_eligible'], text_col].fillna('').astype(str).tolist()
    texts = _clean_social_text(texts_raw)

    if mode == 'ensemble':
        wf, wr = ensemble_weights
        f_spec = MODEL_REGISTRY['finance']
        r_spec = MODEL_REGISTRY['reddit']
        try:
            f = _predict_scores(texts, f_spec.name, f_spec.neg_label, f_spec.pos_label,
                                batch_size=batch_size, max_length=max_seq_len, use_fp16=use_fp16)
        except Exception as e:
            print(f"[warn] finance model failed: {e}; using zeros")
            f = np.zeros(len(texts), dtype=np.float32)
        try:
            r = _predict_scores(texts, r_spec.name, r_spec.neg_label, r_spec.pos_label,
                                batch_size=batch_size, max_length=max_seq_len, use_fp16=use_fp16)
        except Exception as e:
            print(f"[warn] reddit model failed: {e}; falling back to finance only")
            s = f
        else:
            s = wf * f + wr * r
    else:
        key = mode if mode in MODEL_REGISTRY else 'reddit'
        spec = MODEL_REGISTRY[key]
        try:
            s = _predict_scores(texts, spec.name, spec.neg_label, spec.pos_label,
                                batch_size=batch_size, max_length=max_seq_len, use_fp16=use_fp16)
        except Exception as e:
            print(f"[warn] {mode} model failed: {e}; falling back to finance")
            f_spec = MODEL_REGISTRY['finance']
            s = _predict_scores(texts, f_spec.name, f_spec.neg_label, f_spec.pos_label,
                                 batch_size=batch_size, max_length=max_seq_len, use_fp16=use_fp16)

    # Optional VADER blending
    if use_vader or mode == 'reddit_plus':
        v = _vader_scores(texts)
        w = np.clip(vader_weight if use_vader else 0.10, 0.0, 0.49)
        s = (1.0 - w) * s + w * v

    # Intensity heuristics + engagement scaling for reddit_plus
    if mode == 'reddit_plus':
        factors = _intensity_heuristic(texts_raw)
        s = s * factors
        ups = df.loc[df['is_sentiment_eligible'], 'score'].to_numpy(dtype=np.float32)
        ncom = df.loc[df['is_sentiment_eligible'], 'num_comments'].to_numpy(dtype=np.float32)
        s = _engagement_boost(s, ups, ncom, alpha_comments=alpha_comments)

    # Clamp to [-1, 1]
    s = np.clip(s, -1.0, 1.0)
    out = df.copy()
    out.loc[df['is_sentiment_eligible'], 'sent_score'] = s.astype(np.float32)
    out.loc[~df['is_sentiment_eligible'], 'sent_score'] = np.nan
    return out

# ============================
# Monthly aggregation
# ============================

def aggregate_monthly(df: pd.DataFrame, alpha_comments: float = ALPHA_COMMENTS) -> pd.DataFrame:
    """
    Aggregate post-level data into monthly ticker-level metrics.

    The aggregation produces columns:
      * `n_posts`: total posts collected per ticker/month.
      * `mentions_textual`: number of posts used for sentiment (i.e., not media).
      * `mentions_per_day`: daily normalised post count.
      * `attention_z`: z‑score of textual mentions over a 12‑month window.
      * `sent_mean`, `sent_median`: average and median sentiment for the month.
      * `sent_ew_weighted`: engagement‑weighted average sentiment.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with per‑post data and `sent_score` column.
    alpha_comments : float, optional
        Engagement weighting factor for the monthly engagement weight.

    Returns
    -------
    pd.DataFrame
        Aggregated monthly DataFrame.
    """
    if df.empty:
        return pd.DataFrame(columns=[
            'ticker','month','n_posts','mentions_textual','mentions_per_day',
            'attention_z','sent_mean','sent_ew_weighted','sent_median'
        ])
    # Convert to UTC and period month
    month = df['date'].dt.tz_convert('UTC').dt.tz_localize(None).dt.to_period('M')
    dd = df.assign(month=month)

    # Engagement weight per post
    w = np.log1p(dd['score'].clip(lower=0)) + alpha_comments * np.log1p(dd['num_comments'].clip(lower=0))
    dd['eng_w'] = w.fillna(0.0).astype(np.float32)

    # Counts
    grp = dd.groupby(['ticker','month'], as_index=False)
    g_count = grp.size().rename(columns={'size': 'n_posts'})
    
    # Count textual mentions (non-media posts) - FIXED VERSION
    g_textual = dd.groupby(['ticker','month'])['is_media'].apply(lambda x: (~x).sum()).reset_index()
    g_textual.columns = ['ticker', 'month', 'mentions_textual']

    # Per-day normalisation
    days = dd[['ticker','month','date']].copy()
    days['days_in_month'] = days['month'].dt.days_in_month
    days = days.drop_duplicates(['ticker','month'])
    per_day = g_count.merge(days[['ticker','month','days_in_month']], on=['ticker','month'], how='left')
    per_day['mentions_per_day'] = per_day['n_posts'] / per_day['days_in_month']
    per_day = per_day[['ticker','month','mentions_per_day']]

    # Sentiment aggregates - FIXED VERSION
    def safe_weighted_mean(group):
        """Safely compute engagement-weighted mean for a group"""
        sent_scores = group['sent_score'].dropna()
        weights = group.loc[sent_scores.index, 'eng_w']
        
        if len(sent_scores) == 0:
            return np.nan
        if len(weights) == 0 or weights.sum() == 0:
            return sent_scores.mean()
        
        # Align weights with scores
        aligned_weights = weights.reindex(sent_scores.index).fillna(0)
        if aligned_weights.sum() == 0:
            return sent_scores.mean()
        
        return np.average(sent_scores, weights=aligned_weights)
    
    sent = dd.groupby(['ticker','month']).agg(
        sent_mean=('sent_score','mean'),
        sent_median=('sent_score','median'),
    ).reset_index()
    
    # Compute engagement weighted separately
    sent_ew = dd.groupby(['ticker','month']).apply(safe_weighted_mean, include_groups=False).reset_index()
    sent_ew.columns = ['ticker', 'month', 'sent_ew_weighted']
    
    sent = sent.merge(sent_ew, on=['ticker','month'], how='left')

    # Attention z-score over 12 months per ticker using textual mentions
    att = g_textual.copy()
    def _zscore(x: pd.Series) -> pd.Series:
        return (x - x.rolling(12, min_periods=3).mean()) / (x.rolling(12, min_periods=3).std(ddof=0) + 1e-9)
    att['attention_z'] = att.groupby('ticker')['mentions_textual'].transform(_zscore)

    out = (g_count
           .merge(g_textual, on=['ticker','month'], how='left')
           .merge(per_day, on=['ticker','month'], how='left')
           .merge(att[['ticker','month','attention_z']], on=['ticker','month'], how='left')
           .merge(sent, on=['ticker','month'], how='left'))

    out = out.sort_values(['ticker','month']).reset_index(drop=True)
    out['month'] = out['month'].astype(str)
    return out

# ============================
# Pipeline runner
# ============================

def run_pipeline_classic():
    """
    Run the complete pipeline using the classic parameter block. This
    function collects Reddit posts, applies optional company context
    filters, builds sentiment text and eligibility flags, scores
    sentiment, aggregates to monthly metrics, and saves results to
    disk.

    The function prints high‑level coverage statistics upon
    completion.
    """
    tickers = TICKERS
    subs = SUBS
    os.makedirs(OUT_DIR, exist_ok=True)

    # 1) Collect posts
    df = collect_reddit_monthly_scroll(
        tickers=tickers,
        start_year=START_YEAR,
        end_year=END_YEAR,
        subreddits=subs,
        include_ticker_subs=INCLUDE_TICKER_SUBS,
        ticker_sub_map=TICKER_SUB_MAP_DEFAULT,
        per_query_limit=PER_QUERY_LIMIT,
        strict_match=STRICT_MATCH,
        include_top_comments=INCLUDE_TOP_COMMENTS,
        top_comments_k=TOP_COMMENTS_K,
        request_timeout=REQUEST_TIMEOUT,
        verbose=True,
    )

    # Save raw posts
    raw_path = os.path.join(OUT_DIR, 'reddit_posts_all.csv')
    df.to_csv(raw_path, index=False)

    # 2) Optional company context filter
    if APPLY_COMPANY_CONTEXT_FOR:
        df = apply_company_context_filter(
            df,
            only_company_for=APPLY_COMPANY_CONTEXT_FOR,
            finance_words=FINANCE_WORDS,
            product_words_map=PRODUCT_WORDS_MAP,
            finance_subs=tuple(SUBS),
        )

    # 3) Build sentiment text and eligibility
    df = make_sentiment_set(
        df,
        min_headline_chars=MIN_HEADLINE_CHARS,
        min_total_chars=MIN_TOTAL_CHARS,
        include_top_comments=INCLUDE_TOP_COMMENTS,
        max_selftext_chars=MAX_ST_CHARS,
        max_comments_chars=MAX_COMMENTS_CHARS,
    )

    # Save textual posts (eligible and ineligible) for inspection
    textual_path = os.path.join(OUT_DIR, 'reddit_posts_textual.csv')
    df.to_csv(textual_path, index=False)

    # 4) Score sentiment on eligible rows
    df_scored = score_sentiment(
        df,
        text_col='text_for_sentiment',
        mode=MODEL,
        ensemble_weights=ENSEMBLE_WEIGHTS,
        batch_size=BATCH_SIZE,
        max_seq_len=MAX_SEQ_LEN,
        use_fp16=USE_FP16,
        use_vader=USE_VADER,
        vader_weight=VADER_WEIGHT,
        alpha_comments=ALPHA_COMMENTS,
    )

    # Save scored posts
    scored_path = os.path.join(OUT_DIR, 'reddit_posts_scored.csv')
    df_scored.to_csv(scored_path, index=False)

    # 5) Aggregate monthly
    monthly = aggregate_monthly(df_scored, alpha_comments=ALPHA_COMMENTS)

    # Save monthly summary
    monthly_path = os.path.join(OUT_DIR, 'reddit_monthly_sentiment.csv')
    monthly.to_csv(monthly_path, index=False)

    # Print summary
    print(f"Saved raw posts to: {raw_path}  (rows: {len(df):,})")
    print(f"Saved textual posts to: {textual_path}  (rows: {len(df):,})")
    print(f"Saved scored posts to: {scored_path}  (rows: {len(df_scored):,})")
    print(f"Saved monthly summary to: {monthly_path} (rows: {len(monthly):,})")

    # Coverage statistics
    if not monthly.empty:
        month_range_start = pd.Timestamp(year=START_YEAR, month=1, day=1, tz='UTC').to_period('M')
        month_range_end = (pd.Timestamp.utcnow() if END_YEAR is None else pd.Timestamp(year=END_YEAR, month=12, day=31, tz='UTC')).to_period('M')
        expected_months = int(month_range_end - month_range_start) + 1
        print(f"\nUniverse summary → tickers: {len(monthly['ticker'].unique())}, months: {monthly['month'].nunique()}")
        print(f"Date range: {df['date'].min()}  →  {df['date'].max()}")
        cov = (monthly.groupby('ticker')
                     .agg(months_covered=('month','nunique'), total_posts=('ticker','size'))
                     .reset_index())
        cov['coverage_pct'] = 100.0 * cov['months_covered'] / max(expected_months, 1)
        cov['avg_posts_per_month'] = cov['total_posts'] / cov['months_covered'].replace(0, np.nan)
        cov = cov.sort_values(['months_covered','total_posts'], ascending=[False, False])
        print("\nCoverage by ticker:")
        print(cov.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
        # Count months with fewer than five tickers
        breadth = monthly.groupby('month')['ticker'].nunique()
        print(f"\nMonths with <5 tickers: {(breadth < 5).sum()}")

if __name__ == '__main__':
    run_pipeline_classic()

[Scroll] 34 subs | window 2015-01-01..2025-09-17
[Scroll] Tickers: NVDA, AMD, MSFT, ASML, GOOG, GOOGL, PLTR, MRVL, APP, MU, IONQ, RGTI, QBTS, SMR, AI, RDDT, ARBE, AEM, VERU, QQQ
[Scroll] r/agnicoeagle error: received 404 HTTP response
[Scroll] r/agnicoeagle: no matches in window (or all filtered)


/var/folders/lj/jghsddqd5mxbhxdb6mwg7qcw0000gn/T/ipykernel_1180/1220588359.py:359: UserWarning: The comments for this submission have already been fetched, so the updated comment_sort will not have any effect.
  submission.comment_sort = sort


KeyboardInterrupt: 

In [ ]:
# SIMPLIFIED RUNNER USING UTILE_REDDIT.PY
# =====================================

import os
from utile_reddit import run_pipeline_classic

# Define your ticker universe here. These tickers will be used for
# matching posts and generating sentiment metrics.
TICKERS = [
    "NVDA", "AMD", "MSFT", "ASML", "GOOG", "GOOGL", "PLTR", "MRVL", "APP",
    "MU", "IONQ", "RGTI", "QBTS", "SMR", "AI", "RDDT", "ARBE", "AEM", "VERU", "QQQ",
]

# Core finance subreddits plus a few niche ones for better coverage
SUBS = [
    # Original subs
    "stocks", "investing", "stockmarket", "financialnews", "quant",
    "pennystocks", "biotechstocks", "biotech", "mining", "gold", "canadianinvestor",
    
    # Additional general finance subs for better coverage
    "SecurityAnalysis", "ValueInvesting", "options", "wallstreetbets",
    "StockMarket", "investing_discussion", "stocks_penny", "smallcapstocks",
    
    # Sector-specific subs
    "semiconductors", "tech", "technology", "artificial", "MachineLearning",
    "nuclear", "energy", "uranium", "UraniumSqueeze",
    "silverbugs", "precious_metals", "pharma",
]

# Configuration parameters
START_YEAR = 2015
END_YEAR = None
PER_QUERY_LIMIT = 120
INCLUDE_TOP_COMMENTS = True
TOP_COMMENTS_K = 5
MODEL = "reddit_plus"
OUT_DIR = "out/reddit"

print(f"✅ Configuration loaded:")
print(f"   Tickers: {len(TICKERS)}")
print(f"   Subreddits: {len(SUBS)}")
print(f"   Date range: {START_YEAR} to present")

# Run the pipeline
if __name__ == '__main__':
    run_pipeline_classic(
        tickers=TICKERS,
        subs=SUBS,
        start_year=START_YEAR,
        end_year=END_YEAR,
        per_query_limit=PER_QUERY_LIMIT,
        include_top_comments=INCLUDE_TOP_COMMENTS,
        top_comments_k=TOP_COMMENTS_K,
        model=MODEL,
        out_dir=OUT_DIR
    )


In [ ]:
monthly = pd.read_csv('out/reddit/reddit_monthly_sentiment.csv')
monthly

,ticker,month,n_posts,mentions_textual,mentions_per_day,attention_z,sent_mean,sent_median,sent_ew_weighted
0,AEM,2025-07,1,0,0.032258,NaN,0.716375,0.716375,0.716375
1,AEM,2025-08,3,2,0.096774,NaN,0.470834,0.426476,0.477088
2,AEM,2025-09,2,2,0.066667,0.707107,0.447668,0.447668,0.447076
3,AI,2020-12,1,1,0.032258,NaN,0.730249,0.730249,0.730249
4,AI,2021-03,33,28,1.064516,NaN,0.394770,0.641883,0.402792
...,...,...,...,...,...,...,...,...,...
546,VERU,2023-04,8,8,0.266667,-0.894427,0.135556,0.132233,0.183545
547,VERU,2023-05,13,13,0.419355,-0.319636,0.129137,0.106987,0.102318
548,VERU,2023-06,9,9,0.300000,-0.963585,0.080388,0.075281,0.126025
549,VERU,2023-08,2,2,0.064516,-1.537325,0.573648,0.573648,0.526275


In [ ]:
# Print summary statistics
print(f"\nMonthly data summary:")
print(f"Tickers: {monthly['ticker'].nunique()}")
print(f"Months covered: {monthly['month'].nunique()}")
print(f"Date range: {monthly['month'].min()} to {monthly['month'].max()}")

print(f"\nPosts per ticker per month (mean):")
coverage = monthly.groupby('ticker').agg(
    months=('month', 'nunique'),
    avg_posts=('n_posts', 'mean'),
    total_posts=('n_posts', 'sum')
).sort_values('total_posts', ascending=False)
print(coverage.head(10))

print(f"\nSentiment score ranges:")
sent_cols = ['sent_mean', 'sent_median', 'sent_ew_weighted']
for col in sent_cols:
    if col in monthly.columns:
        print(f"{col}: {monthly[col].min():.3f} to {monthly[col].max():.3f} (mean: {monthly[col].mean():.3f})")



Monthly data summary:
Tickers: 19
Months covered: 100
Date range: 2015-01 to 2025-09

Posts per ticker per month (mean):
        months  avg_posts  total_posts
ticker                                
MSFT        37  27.621622         1022
NVDA        33  30.000000          990
QBTS        50  18.320000          916
IONQ        53  16.509434          875
AMD         16  53.187500          851
RGTI        19  39.368421          748
PLTR        20  35.000000          700
AI          51  13.313725          679
RDDT        38  13.763158          523
ARBE         9  51.666667          465

Sentiment score ranges:
sent_mean: -1.000 to 1.000 (mean: 0.169)
sent_median: -1.000 to 1.000 (mean: 0.167)
sent_ew_weighted: -1.000 to 1.000 (mean: 0.174)


In [ ]:
monthly

,ticker,month,n_posts,mentions_textual,mentions_per_day,attention_z,sent_mean,sent_median,sent_ew_weighted
0,AEM,2025-07,1,0,0.032258,NaN,0.716375,0.716375,0.716375
1,AEM,2025-08,3,2,0.096774,NaN,0.470834,0.426476,0.477088
2,AEM,2025-09,2,2,0.066667,0.707107,0.447668,0.447668,0.447076
3,AI,2020-12,1,1,0.032258,NaN,0.730249,0.730249,0.730249
4,AI,2021-03,33,28,1.064516,NaN,0.394770,0.641883,0.402792
...,...,...,...,...,...,...,...,...,...
546,VERU,2023-04,8,8,0.266667,-0.894427,0.135556,0.132233,0.183545
547,VERU,2023-05,13,13,0.419355,-0.319636,0.129137,0.106987,0.102318
548,VERU,2023-06,9,9,0.300000,-0.963585,0.080388,0.075281,0.126025
549,VERU,2023-08,2,2,0.064516,-1.537325,0.573648,0.573648,0.526275


Reddit API credentials set successfully


### Join Reddit monthly sentiment with monthly log returns — co-movement check (not predictive)

In [ ]:



def _load_sentiment(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # month may be string 'YYYY-MM' — normalize to Period[M]
    if 'month' in df.columns:
        df['month'] = pd.PeriodIndex(df['month'].astype(str), freq='M')
    keep = ['ticker','month','mentions_per_day','attention_z','sent_mean','sent_ew_weighted','sent_median']
    missing = [c for c in keep if c not in df.columns]
    if missing:
        raise ValueError(f"Sentiment file missing columns: {missing}")
    return df[keep].copy()


def _load_prices_daily(path: str, date_col: str = 'date') -> pd.DataFrame:
    dfp = pd.read_csv(path)
    cols = {c.lower(): c for c in dfp.columns}
    if 'ticker' not in cols:
        raise ValueError("Prices need a 'ticker' column")
    px_col = cols.get('adj_close') or cols.get('close')
    if px_col is None:
        raise ValueError("Prices need 'adj_close' or 'close' column")
    dcol = cols.get(date_col.lower(), None)
    if dcol is None:
        # try to find a likely date column
        for cand in ['date','datetime','timestamp']:
            if cand in cols:
                dcol = cols[cand]
                break
    if dcol is None:
        raise ValueError("Could not locate date column in prices")

    dfp = dfp.rename(columns={cols['ticker']: 'ticker', dcol: 'date', px_col: 'close'})
    dfp['date'] = pd.to_datetime(dfp['date'], errors='coerce')
    dfp = dfp.dropna(subset=['date','ticker','close'])

    # Monthly last close per ticker
    dfp['month'] = dfp['date'].dt.to_period('M')
    monthly = (
        dfp.sort_values(['ticker','date'])
           .groupby(['ticker','month'], as_index=False)
           .agg(last_close=('close','last'))
    )
    # Log returns within month -> same-month return is change *within* the month (co-movement)
    monthly['log_close'] = np.log(monthly['last_close'])
    monthly['logret'] = monthly.groupby('ticker')['log_close'].diff()
    monthly = monthly.drop(columns=['log_close'])
    return monthly


def _load_prices_monthly(path: str) -> pd.DataFrame:
    dfm = pd.read_csv(path)
    cols = {c.lower(): c for c in dfm.columns}
    if 'ticker' not in cols:
        raise ValueError("Monthly prices need a 'ticker' column")
    # allow 'month' or 'date'
    if 'month' in cols:
        mcol = cols['month']
    elif 'date' in cols:
        mcol = cols['date']
    else:
        raise ValueError("Need a 'month' or 'date' column for monthly prices")
    px_col = cols.get('adj_close') or cols.get('close') or cols.get('price')
    if px_col is None:
        raise ValueError("Monthly prices need 'adj_close'/'close'/'price' column")

    dfm = dfm.rename(columns={cols['ticker']:'ticker', mcol:'month', px_col:'close'})
    dfm['month'] = pd.PeriodIndex(pd.to_datetime(dfm['month'], errors='coerce').dt.to_period('M'), freq='M')
    dfm = dfm.dropna(subset=['month','ticker','close'])
    dfm = dfm.sort_values(['ticker','month'])
    dfm['logret'] = np.log(dfm['close']).groupby(dfm['ticker']).diff()
    return dfm[['ticker','month','logret']]


def safe_corr(x: pd.Series, y: pd.Series) -> float:
    a = pd.concat([x, y], axis=1).dropna()
    if len(a) < 2:
        return np.nan
    return float(a.iloc[:,0].corr(a.iloc[:,1]))


def main(sent_path: str, prices_path: str, prices_monthly: bool, out_path: str) -> None:
    sent = _load_sentiment(sent_path)
    if prices_monthly:
        px = _load_prices_monthly(prices_path)
    else:
        px = _load_prices_daily(prices_path)

    # Align and join on (ticker, month)
    merged = sent.merge(px[['ticker','month','logret']], on=['ticker','month'], how='inner')

    # Per-ticker co-movement correlations (same month)
    rows = []
    for tkr, g in merged.groupby('ticker'):
        rows.append({
            'ticker': tkr,
            'N_months': int(g[['month']].dropna().shape[0]),
            'corr(logret, attention_z)': safe_corr(g['logret'], g['attention_z']),
            'corr(logret, mentions_per_day)': safe_corr(g['logret'], g['mentions_per_day']),
            'corr(logret, sent_ew_weighted)': safe_corr(g['logret'], g['sent_ew_weighted']),
            'corr(logret, sent_mean)': safe_corr(g['logret'], g['sent_mean']),
            'corr(logret, sent_median)': safe_corr(g['logret'], g['sent_median']),
        })
    per_ticker = pd.DataFrame(rows).sort_values('ticker').reset_index(drop=True)

    # Pooled correlations (all tickers together)
    pooled = {
        'pooled_corr(logret, attention_z)': safe_corr(merged['logret'], merged['attention_z']),
        'pooled_corr(logret, mentions_per_day)': safe_corr(merged['logret'], merged['mentions_per_day']),
        'pooled_corr(logret, sent_ew_weighted)': safe_corr(merged['logret'], merged['sent_ew_weighted']),
        'pooled_corr(logret, sent_mean)': safe_corr(merged['logret'], merged['sent_mean']),
        'pooled_corr(logret, sent_median)': safe_corr(merged['logret'], merged['sent_median']),
    }

    # Save merged (handy for spot checks) and print summaries
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    merged.to_csv(out_path, index=False)

    print("Saved merged same-month data →", out_path)
    print("\nPer-ticker same-month Pearson correlations (co-movement):")
    print(per_ticker.to_string(index=False, float_format=lambda v: f"{v: .3f}"))

    print("\nPooled same-month Pearson correlations:")
    for k, v in pooled.items():
        print(f"  {k}: {v:.3f}")


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description='Join monthly Reddit sentiment with monthly log returns (co-movement check)')
    ap.add_argument('--sent', type=str, default='out/reddit/reddit_monthly_sentiment.csv')
    ap.add_argument('--prices', type=str, required=True, help='Path to daily or monthly prices CSV')
    ap.add_argument('--prices-monthly', action='store_true', help='Set if prices CSV is already monthly')
    ap.add_argument('--out', type=str, default='out/reddit/sentiment_returns_same_month.csv')
    args = ap.parse_args()

    main(args.sent, args.prices, args.prices_monthly, args.out)
